In [1]:
from pathlib import Path
import pandas as pd

DATA = Path("../Data/Jersey_Birth_Death.xlsx")  # needs openpyxl: pip install openpyxl

# Columns A, D and E: period plus the complete births/deaths series.
# B and C are only filled for Dec-24 and Dec-25, so I skip them.
raw = pd.read_excel(DATA, usecols=[0, 3, 4], header=0,
                    names=["period", "Births", "Deaths"])

# Drop the Dec-25 row with blanks in D:E, then any remaining duplicate periods
raw = raw.dropna(subset=["Births", "Deaths"]).drop_duplicates("period", keep="last")

# "Dec-00" is either text or an Excel date, so handle both
period = pd.to_datetime(raw["period"], format="%b-%y", errors="coerce")
period = period.fillna(pd.to_datetime(raw["period"], errors="coerce"))

df = pd.DataFrame({
    "date": pd.to_datetime(period.dt.year.astype(str) + "-01-01"),  # Dec-YY -> that year
    "Births": raw["Births"].astype(int).to_numpy(),
    "Deaths": raw["Deaths"].astype(int).to_numpy(),
}).sort_values("date").reset_index(drop=True)

# Sanity checks: catch a parsing problem before it reaches the chart
assert df["date"].dt.year.tolist() == list(range(2000, 2026)), "years look wrong"
assert df.loc[df["date"].dt.year == 2012, "Births"].iloc[0] == 1130
assert df.iloc[-1][["Births", "Deaths"]].tolist() == [710, 910]
df.tail(5)

,date,Births,Deaths
21,2021-01-01,900,820
22,2022-01-01,840,940
23,2023-01-01,800,920
24,2024-01-01,730,890
25,2025-01-01,710,910


In [13]:
# ---- Chart -----------------------------------------------------------------------
Y_TITLE = "Births and deaths per year"
y_scale = alt.Scale(domain=[600, 1200])

x = alt.X("date:T", title=None, axis=alt.Axis(format="%Y", tickCount=6))

def y(field):
    return alt.Y(f"{field}:Q", title=Y_TITLE, scale=y_scale)

colour = alt.Color("series:N", legend=None)  # theme assigns the palette

def tag(chart, series):
    """Give a layer a constant 'series' so it picks up that series' theme colour."""
    return chart.transform_calculate(series=f"'{series}'")

def shade(condition, series):
    return tag(alt.Chart(monthly).transform_filter(condition), series).mark_area(
        opacity=0.18
    ).encode(x=x, y=y("Deaths"), y2="Births:Q", color=colour)

area_up = shade("datum.Births >= datum.Deaths", "Births")
area_down = shade("datum.Births < datum.Deaths", "Deaths")

lines = alt.Chart(long).mark_line(strokeWidth=1.8).encode(
    x=x, y=y("value"), color=colour
)

# End-of-line direct labels instead of a legend
labels = alt.Chart(long).mark_text(align="left", dx=6, fontWeight="bold").encode(
    x=alt.X("max(date):T", title=None),
    y=alt.Y("value:Q", aggregate={"argmax": "date"}, title=Y_TITLE, scale=y_scale),
    text="series:N",
    color=colour,
)

chart = (area_up + area_down + lines + labels).properties(
    width=700, height=270,
    autosize=alt.AutoSizeParams(type="fit", contains="padding"),
)

styles.save(chart, path=".", name="jersey_births_deaths_2000_2025", svg=True)
chart

alt.LayerChart(...)

In [14]:
import inspect
from PIL import Image

print(inspect.signature(styles.save))   # look for scale / ppi / width / height args
print(inspect.getsource(styles.save))   # shows what it does to the chart before exporting

print(Image.open("jersey_births_deaths_2000_2025.png").size)  # actual exported pixels

(*args, **kwargs)
    def save(self, *args, **kwargs):
        """Save chart to file(s). See utils.file_operations.save_chart for details."""
        return save_chart(*args, **kwargs)

(1400, 1120)


In [16]:
chart.save("jersey_births_deaths_2000_2025.png", scale_factor=4)  # 2800 x 1080, sharp when scaled down
chart.save("jersey_births_deaths_2000_2025.svg")                  # vector, sharp at any size